In [1]:
# gsheets_consolidate_local.py

import io
import re
import requests
from datetime import datetime
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', None)
pd.options.display.float_format = '{:,.2f}'.format

from warnings import filterwarnings
filterwarnings('ignore')

import Helper as helper  # твой модуль

today_date = datetime.today().strftime("%Y%m%d")

# ---------------------- источники ----------------------
sheet_name = 'Займы'

office_urls = {
    "Кропоткина":     "https://docs.google.com/spreadsheets/d/1IHUDHHVbu4tDDMOM5lRrpZqqWjzaSNJ5BZxGOTYFLFY/export?format=xlsx",
    "Станиславского": "yadisk://https://disk.yandex.ru/i/iTHeN9bxZbQw5g",  # Яндекс.Диск (публичная ссылка)
    "Барнаул":        "https://docs.google.com/spreadsheets/d/13Jki90mHVKZzWoE6dYKViw46z_GsuJJZriQDh0_1UXQ/export?format=xlsx",
    "Новокузнецк":    "https://docs.google.com/spreadsheets/d/1ymloR8HFIzHjUNydS4jz-8FNCQxGE1_gnJG8vW3oEe0/export?format=xlsx",
}

# ---------------------- загрузка ----------------------
def _download_yadisk_public(public_url: str, timeout: int = 60) -> bytes:
    """Получить bytes файла по публичной ссылке Яндекс.Диска."""
    api = "https://cloud-api.yandex.net/v1/disk/public/resources/download"
    r = requests.get(api, params={"public_key": public_url}, timeout=timeout)
    r.raise_for_status()
    data = r.json()
    if "href" not in data:
        raise RuntimeError(f"Не удалось получить href от Я.Диска: {data}")
    f = requests.get(data["href"], timeout=timeout)
    f.raise_for_status()
    return f.content

def _read_excel_from_bytes(raw: bytes, sheet: str | int | None = None) -> pd.DataFrame:
    with io.BytesIO(raw) as bio:
        return pd.read_excel(bio, sheet_name=sheet)

def _load_office_df(office: str, url: str, sheet: str) -> pd.DataFrame:
    """
    Для Станиславского — скачиваем с Я.Диска.
    Для остальных — читаем напрямую из экспорт-ссылки Google Sheets (xlsx).
    """
    if url.startswith("yadisk://"):
        public_url = url.replace("yadisk://", "", 1)
        raw = _download_yadisk_public(public_url)
        # простая эвристика: xlsx-zip обычно начинается с PK
        if raw[:2] != b"PK":
            # если вдруг это csv — можно адаптировать при необходимости
            raise RuntimeError("Ожидался XLSX на Я.Диске, но сигнатура не похожа на XLSX (PK).")
        df = _read_excel_from_bytes(raw, sheet)
    else:
        df = pd.read_excel(url, sheet_name=sheet)

    df = helper.start_col(df)  # твоя предочистка
    df["филиал"] = office
    return df

frames = []
for office, url in office_urls.items():
    df_i = _load_office_df(office, url, sheet_name)
    frames.append(df_i)

# ---------------- гармонизация колонок -----------------
def _norm_col(s: str) -> str:
    s = str(s).strip().lower()
    s = s.replace('\n', ' ')
    s = re.sub(r'\s*/\s*', ' / ', s)
    s = re.sub(r'\s+', ' ', s)
    s = re.sub(r'\s*\(.*?\)\s*', '', s)
    return s.strip()

synonyms = {
    # базовые
    "филиал": "филиал",
    "клиент (фио)": "клиент",
    "клиент": "клиент",
    "залог (марка, модель)": "залог",
    "залог": "залог",
    # даты
    "дата выдачи": "дата_выдачи",
    "плановая дата платежа": "плановая_дата_платежа",
    # деньги/ставки
    "сумма займа, руб.": "сумма_займа",
    "сумма займа": "сумма_займа",
    "ставка, % в месяц": "ставка_в_мес",
    "ставка % в месяц": "ставка_в_мес",
    # статусы
    "статус": "статус",
    "платеж по графику": "платеж_по_графику",
    "проценты за месяц, руб.": "проценты_за_месяц",
    # просрочка/интервалы
    "пз, дни": "пз_дни",
    "интервал пз, дни": "интервал_пз_дни",
    # оплаты/штрафы
    "оплачено процентов / основного долга": "оплачено_процентов_или_тело",
    "оплачено штрафов / комиссий": "оплачено_штрафов_комиссий",
    # прочее
    "комментарий": "комментарий",
}

def harmonize_columns(df: pd.DataFrame) -> pd.DataFrame:
    rename_map = {c: synonyms.get(_norm_col(c), _norm_col(c)) for c in df.columns}
    out = df.rename(columns=rename_map).copy()
    empty_cols = [c for c in out.columns if out[c].isna().all()]
    return out.drop(columns=empty_cols, errors='ignore')

# ---------------- консолидация -----------------
def consolidate_frames(frames: list) -> pd.DataFrame:
    aligned = [harmonize_columns(f) for f in frames]
    result = pd.concat(aligned, ignore_index=True, sort=True)

    preferred = [
        "филиал",
        "клиент", "залог",
        "дата_выдачи", "плановая_дата_платежа", "статус",
        "сумма_займа", "ставка_в_мес", "платеж_по_графику", "проценты_за_месяц",
        "пз_дни", "интервал_пз_дни",
        "оплачено_процентов_или_тело", "оплачено_штрафов_комиссий",
        "комментарий",
    ]
    existing_pref = [c for c in preferred if c in result.columns]
    the_rest = [c for c in result.columns if c not in existing_pref]
    return result[existing_pref + the_rest]

df = consolidate_frames(frames)
df.head(3)


,филиал,клиент,залог,дата_выдачи,плановая_дата_платежа,статус,сумма_займа,ставка_в_мес,платеж_по_графику,проценты_за_месяц,пз_дни,интервал_пз_дни,оплачено_процентов_или_тело,оплачено_штрафов_комиссий,комментарий,unnamed: 14,unnamed: 15,unnamed: 16,unnamed: 17
0,Кропоткина,Алыйев Авыл Гасан Оглы,NaN,2025-07-04,2025-10-04,оплата по графику,400000,6.90,NaN,"27,600.00",NaN,NaN,"84,300.00",NaN,NaN,4,NaT,NaN,NaT
1,Кропоткина,Комаров Михаил Андреевич,"Парковка машино-место, площадь: 15 ул. Кирова,...",2025-07-04,2025-10-04,оплата по графику,4500000,6.00,NaN,"270,000.00",NaN,NaN,"810,000.00",NaN,NaN,4,NaT,NaN,NaT
2,Кропоткина,Скиба Иван Александрович,TOYOTA CAMRY СЕРО-КОРИЧНЕВЫЙ МЕТАЛЛИК 2018 г.в.,2025-07-08,2025-10-08,оплата по графику,1500000,6.50,NaN,"97,500.00",NaN,0-15 дней,"97,500.00","2,500.00",NaN,8,NaT,NaN,NaT


In [2]:
# ================= НОРМАЛИЗАЦИЯ ДАННЫХ В df =================
import numpy as np
import re
from datetime import datetime

# --- функции-преобразователи ---
def normalize_money(x):
    """Деньги -> float: убирает ₽/руб/пробелы, запятые -> точка, оставляет десятичную часть."""
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    if s == "":
        return np.nan
    s = re.sub(r"(руб(лей|ля|\.?)|₽|RUB|rub)", "", s, flags=re.IGNORECASE)
    s = s.replace("\xa0", " ").replace(" ", "")
    s = s.replace(",", ".")
    if s.count(".") > 1:                       # убираем «тысячные» точки
        parts = s.split(".")
        s = "".join(parts[:-1]) + "." + parts[-1]
    m = re.search(r"\d+(?:\.\d+)?", s)
    return float(m.group()) if m else np.nan

def normalize_percent(x):
    """Проценты -> число в %: понимает '5', '5%', '5,5', '0,03' => 3.0."""
    if pd.isna(x):
        return np.nan
    s = str(x).strip().replace("\xa0", " ")
    s = s.replace("%", "").replace(" ", "").replace(",", ".")
    try:
        v = float(s)
        return v * 100 if v <= 1 else v
    except Exception:
        return np.nan

def parse_date_safe(x):
    """Дата с приоритетом dayfirst. Нераспознаваемые -> NaT."""
    if pd.isna(x):
        return pd.NaT
    if isinstance(x, (pd.Timestamp, datetime)):
        return pd.to_datetime(x, errors="coerce")
    s = str(x).strip().replace("\xa0", " ")
    dt = pd.to_datetime(s, dayfirst=True, errors="coerce")
    if pd.isna(dt):
        dt = pd.to_datetime(s, errors="coerce")
    return dt

# --- 1) нормализация категорий (филиал) ---
if "филиал" in df.columns:
    filial_map = {
        # существующие
        "станиславского": "Станиславского",
        "станиславского, офис": "Станиславского",
        "кропоткина": "Кропоткина",
        "кропоткина, офис": "Кропоткина",
        # новые
        "барнаул": "Барнаул",
        "барнаул, офис": "Барнаул",
        "новокузнецк": "Новокузнецк",
        "новокузнецк, офис": "Новокузнецк",
    }
    df["филиал"] = (
        df["филиал"]
        .astype(str).str.strip().str.lower()
        .map(filial_map)
        .fillna(df["филиал"])
    )

# --- 2) деньги ---
money_cols_all = ["сумма_займа", "проценты_за_месяц", "оплачено_процентов_или_тело", "оплачено_штрафов_комиссий"]
money_cols = [c for c in money_cols_all if c in df.columns]
for c in money_cols:
    df[c] = df[c].apply(normalize_money)

# --- 3) проценты (в процентах) ---
percent_cols_all = ["ставка_в_мес"]
percent_cols = [c for c in percent_cols_all if c in df.columns]
for c in percent_cols:
    df[c] = df[c].apply(normalize_percent)

# --- 4) даты ---
date_cols_all = ["дата_выдачи", "плановая_дата_платежа"]
date_cols = [c for c in date_cols_all if c in df.columns]
for c in date_cols:
    df[c] = df[c].apply(parse_date_safe)

# --- 5) флаги-валидации ---
flags = pd.DataFrame(index=df.index)

if "сумма_займа" in df.columns:
    flags["сумма_займа__le0"] = df["сумма_займа"].notna() & (df["сумма_займа"] <= 0)

if "ставка_в_мес" in df.columns:
    # при необходимости подправьте верхнюю границу
    flags["ставка_в_мес__lt0"]  = df["ставка_в_мес"].notna() & (df["ставка_в_мес"] < 0)
    flags["ставка_в_мес__gt20"] = df["ставка_в_мес"].notna() & (df["ставка_в_мес"] > 20)

for c in date_cols:
    # были исходные значения, но стали NaT
    flags[f"{c}__invalid"] = df[c].isna() & df[c].astype(str).str.strip().ne("")
    # на будущее
    flags[f"{c}__in_future"] = pd.to_datetime(df[c], errors="coerce") > pd.Timestamp.today().normalize()

# --- 6) порядок столбцов (важные вперёд) ---
preferred = [
    "филиал",
    "клиент", "залог",
    "дата_выдачи", "плановая_дата_платежа", "статус",
    "сумма_займа", "ставка_в_мес", "платеж_по_графику", "проценты_за_месяц",
    "пз_дни", "интервал_пз_дни",
    "оплачено_процентов_или_тело", "оплачено_штрафов_комиссий",
    "комментарий",
]
existing_pref = [c for c in preferred if c in df.columns]
the_rest = [c for c in df.columns if c not in existing_pref]
df = df[existing_pref + the_rest]

df.head(2)


,филиал,клиент,залог,дата_выдачи,плановая_дата_платежа,статус,сумма_займа,ставка_в_мес,платеж_по_графику,проценты_за_месяц,пз_дни,интервал_пз_дни,оплачено_процентов_или_тело,оплачено_штрафов_комиссий,комментарий,unnamed: 14,unnamed: 15,unnamed: 16,unnamed: 17
0,Кропоткина,Алыйев Авыл Гасан Оглы,NaN,2025-07-04,2025-10-04,оплата по графику,"400,000.00",6.90,NaN,"27,600.00",NaN,NaN,"84,300.00",NaN,NaN,4,NaT,NaN,NaT
1,Кропоткина,Комаров Михаил Андреевич,"Парковка машино-место, площадь: 15 ул. Кирова,...",2025-07-04,2025-10-04,оплата по графику,"4,500,000.00",6.00,NaN,"270,000.00",NaN,NaN,"810,000.00",NaN,NaN,4,NaT,NaN,NaT


In [3]:
# ==== Выдачи по месяцам: с фильтром по дате (>= 2025-07), самодостаточно ====
import pandas as pd
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# 0) если сводки нет — собираем её из df
if "месяц_итого" not in globals():
    assert "дата_выдачи" in df.columns and "сумма_займа" in df.columns, \
        "Нужно, чтобы в df были колонки 'дата_выдачи' и 'сумма_займа'"
    _tmp = df.copy()
    _tmp["дата_выдачи"] = pd.to_datetime(_tmp["дата_выдачи"], errors="coerce")
    _tmp = _tmp[_tmp["дата_выдачи"].notna()]
    _tmp["период_месяц"] = _tmp["дата_выдачи"].dt.to_period("M").dt.to_timestamp()
    месяц_итого = (
        _tmp.groupby("период_месяц", as_index=False)
            .agg(количество=("сумма_займа", "size"),
                 сумма_займа_итого=("сумма_займа", "sum"))
            .sort_values("период_месяц")
    )

# --- 1) параметры фильтра ---
START_FROM = "2025-07"  # включительно (YYYY-MM)

# --- 2) подготовка дат и фильтрация ---
dfm = месяц_итого.copy()
dfm["period_m"] = pd.to_datetime(dfm["период_месяц"], errors="coerce").dt.to_period("M")
start_p = pd.Period(START_FROM, freq="M")
dfm = dfm[dfm["period_m"] >= start_p].copy()

# метка месяца для категориальной оси
dfm["месяц_метка"] = dfm["period_m"].astype(str)  # YYYY-MM
dfm = dfm.sort_values("месяц_метка")
cats = dfm["месяц_метка"].unique().tolist()

# --- 3) цвета ---
COLOR_SUM = "#4169E1"   # синий
COLOR_CNT = "#DC143C"   # красный

fig = make_subplots(specs=[[{"secondary_y": True}]])

# --- 4) Сумма — столбцы + подписи ---
fig.add_trace(
    go.Bar(
        x=dfm["месяц_метка"],
        y=dfm["сумма_займа_итого"],
        name="Сумма займа (итого)",
        marker_color=COLOR_SUM,
        text=dfm["сумма_займа_итого"].round(0),
        texttemplate="%{text:,.0f}",
        textposition="outside",
        hovertemplate="%{x}<br>Сумма: %{y:,.0f} ₽<extra></extra>",
    ),
    secondary_y=False,
)

# --- 5) Количество — линия + подписи (чередуем право/лево) ---
positions = ["top right" if i % 2 == 0 else "top left" for i in range(len(dfm))]
fig.add_trace(
    go.Scatter(
        x=dfm["месяц_метка"],
        y=dfm["количество"],
        mode="lines+markers+text",
        name="Количество",
        line=dict(color=COLOR_CNT, width=2),
        marker=dict(color=COLOR_CNT),
        text=dfm["количество"],
        texttemplate="%{text:.0f}",
        textposition=positions,
        textfont=dict(size=12),
        cliponaxis=False,
        hovertemplate="%{x}<br>Количество: %{y:.0f}<extra></extra>",
    ),
    secondary_y=True,
)

# --- 6) оформление ---
fig.update_layout(
    title=f"Выдачи по месяцам (с {START_FROM})",
    bargap=0.2,
    hovermode="x unified",
    legend=dict(orientation="h", yanchor="bottom", y=-0.2, xanchor="center", x=0.5),
    margin=dict(t=60, r=20, b=60, l=60),
)
fig.update_xaxes(type="category", categoryorder="array", categoryarray=cats, title_text="Месяц")
fig.update_yaxes(title_text="Сумма, ₽", tickformat=",", secondary_y=False)
fig.update_yaxes(title_text="Количество, шт", tickformat=",.0f", secondary_y=True)

# --- 7) вывод/сохранение ---
fig.show()
fig.write_html("график_выдачи_по_месяцам.html", include_plotlyjs="cdn")
print("Сохранено: график_выдачи_по_месяцам.html")


Сохранено: график_выдачи_по_месяцам.html


In [4]:
# ==== По филиалам: самодостаточно + фильтр по дате (>= 2025-07) ====
import pandas as pd
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# 0) если сводки нет — собираем её из df
if "месяц_по_филиалам" not in globals():
    assert "дата_выдачи" in df.columns and "сумма_займа" in df.columns and "филиал" in df.columns, \
        "В df должны быть 'дата_выдачи', 'сумма_займа', 'филиал'"
    _tmp = df.copy()
    _tmp["дата_выдачи"] = pd.to_datetime(_tmp["дата_выдачи"], errors="coerce")
    _tmp = _tmp[_tmp["дата_выдачи"].notna()]
    _tmp["период_месяц"] = _tmp["дата_выдачи"].dt.to_period("M").dt.to_timestamp()
    месяц_по_филиалам = (
        _tmp.groupby(["период_месяц","филиал"], as_index=False)
            .agg(количество=("сумма_займа","size"),
                 сумма_займа_итого=("сумма_займа","sum"))
            .sort_values(["период_месяц","филиал"])
    )

# --- параметры фильтра ---
START_FROM = "2025-07"  # включительно (YYYY-MM)

# --- подготовка дат и фильтрация ---
dfb = месяц_по_филиалам.copy()
dfb["period_m"] = pd.to_datetime(dfb["период_месяц"], errors="coerce").dt.to_period("M")
start_p = pd.Period(START_FROM, freq="M")
dfb = dfb[dfb["period_m"] >= start_p].copy()

# если после фильтра пусто — аккуратное сообщение и выход
if dfb.empty:
    print(f"Нет данных для отображения (фильтр с {START_FROM}).")
else:
    # метка для категориальной оси
    dfb["месяц_метка"] = dfb["period_m"].astype(str)  # YYYY-MM
    dfb = dfb.sort_values(["месяц_метка", "филиал"])
    cats = dfb["месяц_метка"].unique().tolist()

    # нормализация названий (для устойчивого сопоставления цветов)
    norm = lambda s: str(s).strip().lower()

    # фиксированные цвета для ключевых офисов
    PRIMARY_BLUE = "#4169E1"   # Кропоткина
    PRIMARY_RED  = "#DC143C"   # Станиславского
    fixed = {norm("Кропоткина"): PRIMARY_BLUE, norm("Станиславского"): PRIMARY_RED}

    # стабильная палитра для прочих офисов
    fallback = ["#228B22", "#8A2BE2", "#FF8C00", "#708090", "#2ca02c", "#9467bd", "#17becf"]

    branches = dfb["филиал"].dropna().unique()
    branches_sorted = sorted(branches, key=norm)

    # сопоставление офис -> цвет
    color_map, fi = {}, 0
    for br in branches_sorted:
        key = norm(br)
        if key in fixed:
            color_map[br] = fixed[key]
        else:
            color_map[br] = fallback[fi % len(fallback)]
            fi += 1

    fig = make_subplots(specs=[[{"secondary_y": True}]])

    for br in branches_sorted:
        d = dfb[dfb["филиал"] == br]
        col = color_map[br]
        grp = f"group_{br}"

        # Сумма — столбцы (в легенде)
        fig.add_trace(
            go.Bar(
                x=d["месяц_метка"],
                y=d["сумма_займа_итого"],
                name=f"{br} — сумма",
                marker_color=col,
                text=d["сумма_займа_итого"].round(0),
                texttemplate="%{text:,.0f}",
                textposition="outside",
                hovertemplate="%{x}<br>Филиал: "+br+"<br>Сумма: %{y:,.0f} ₽<extra></extra>",
                legendgroup=grp,
            ),
            secondary_y=False,
        )

        # Количество — линия + подписи (чередуем право/лево), не дублируем в легенде
        positions = ["top right" if i % 2 == 0 else "top left" for i in range(len(d))]
        fig.add_trace(
            go.Scatter(
                x=d["месяц_метка"],
                y=d["количество"],
                mode="lines+markers+text",
                name=f"{br} — количество",
                line=dict(color=col, width=2),
                marker=dict(color=col),
                text=d["количество"],
                texttemplate="%{text:.0f}",
                textposition=positions,
                textfont=dict(size=12),
                cliponaxis=False,
                hovertemplate="%{x}<br>Филиал: "+br+"<br>Количество: %{y:.0f}<extra></extra>",
                legendgroup=grp,
                showlegend=False,
            ),
            secondary_y=True,
        )

    fig.update_layout(
        title=f"Выдачи по месяцам и филиалам (с {START_FROM})",
        barmode="group",
        bargap=0.2,
        hovermode="x unified",
        legend=dict(orientation="h", yanchor="bottom", y=-0.25, xanchor="center", x=0.5),
        margin=dict(t=60, r=20, b=80, l=60),
    )

    # ось X — только отфильтрованные месяцы
    fig.update_xaxes(type="category", categoryorder="array", categoryarray=cats, title_text="Месяц")
    fig.update_yaxes(title_text="Сумма, ₽", tickformat=",", secondary_y=False)
    fig.update_yaxes(title_text="Количество, шт", tickformat=",.0f", secondary_y=True)

    fig.show()

    # (опционально) сохранить HTML
    fig.write_html("график_по_филиалам_по_месяцам.html", include_plotlyjs="cdn")
    print("Сохранено: график_по_филиалам_по_месяцам.html")


Сохранено: график_по_филиалам_по_месяцам.html


In [5]:
# ==== План-факт по неделям (сумма и % выполнения) ====
import pandas as pd
import numpy as np
from datetime import datetime
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# ----- ПАРАМЕТРЫ -----
месяц_план = "2025-10"        # месяц плана (YYYY-MM)
план_месяца = 17_500_000      # плановая сумма за месяц, ₽
фильтр_офиса = None           # None = все офисы; либо строка, например "Кропоткина"

# ----- ПОДГОТОВКА ДАННЫХ -----
wrk = df.copy()
wrk["дата_выдачи"] = pd.to_datetime(wrk["дата_выдачи"], errors="coerce")
wrk = wrk[wrk["дата_выдачи"].notna()]

if фильтр_офиса:
    wrk = wrk[wrk["филиал"].astype(str).str.strip() == фильтр_офиса]

start = pd.Timestamp(месяц_план + "-01")
end   = (start + pd.offsets.MonthEnd(1))  # последнее число месяца (включительно)

# неделя = с понедельника
wrk["неделя_начало"] = (wrk["дата_выдачи"] - pd.to_timedelta(wrk["дата_выдачи"].dt.weekday, unit="D")).dt.normalize()

# факт: берём только выдачи, попавшие в месяц по дате выдачи
mask = (wrk["дата_выдачи"] >= start) & (wrk["дата_выдачи"] <= end)
факт = (wrk.loc[mask]
          .groupby("неделя_начало", as_index=False)["сумма_займа"]
          .sum()
          .rename(columns={"сумма_займа": "факт_сумма"}))

# ----- КОНСТРУКЦИЯ СЕТКИ НЕДЕЛЬ В МЕСЯЦЕ -----
# список всех понедельников, чьи недели пересекаются с рассматриваемым месяцем
пн_до_месяца = (start - pd.to_timedelta(start.weekday(), unit="D")).normalize()
пн = []
cur = пн_до_месяца
while cur <= end:
    пн.append(cur)
    cur = cur + pd.Timedelta(days=7)

сетка = pd.DataFrame({"неделя_начало": пн})

# доля дней каждой недели, приходящаяся на данный месяц → план недели
def дней_пересечения(week_start):
    week_end = week_start + pd.Timedelta(days=6)
    a = max(week_start, start)
    b = min(week_end, end)
    return max((b - a).days + 1, 0)

сетка["дней_в_месяце"] = сетка["неделя_начало"].apply(дней_пересечения)
сетка = сетка[сетка["дней_в_месяце"] > 0].copy()

дней_в_месяце = (end - start).days + 1
сетка["план_сумма"] = (сетка["дней_в_месяце"] / дней_в_месяце) * план_месяца

# ----- СВОДНАЯ ТАБЛИЦА НЕДЕЛЯМИ -----
табл = (сетка.merge(факт, on="неделя_начало", how="left")
              .fillna({"факт_сумма": 0.0})
              .sort_values("неделя_начало"))

табл["выполнение_%"] = np.where(табл["план_сумма"] > 0,
                               100 * табл["факт_сумма"] / табл["план_сумма"],
                               np.nan)

# подписи и категории для оси X
табл["неделя_метка"] = табл["неделя_начало"].dt.strftime("%Y-%m-%d")
cats = табл["неделя_метка"].tolist()

# округления для красивых подписей
табл["план_сумма_окр"] = табл["план_сумма"].round(0)
табл["факт_сумма_окр"] = табл["факт_сумма"].round(0)
табл["выполнение_%_окр"] = табл["выполнение_%"].round(1)

# ----- ВИЗУАЛИЗАЦИЯ -----
fig = make_subplots(specs=[[{"secondary_y": True}]])

# столбцы: план и факт
fig.add_trace(
    go.Bar(
        x=табл["неделя_метка"],
        y=табл["план_сумма"],
        name="План, ₽",
        marker_color="#A9A9A9",  # тёмно-серый
        text=табл["план_сумма_окр"],
        texttemplate="%{text:,.0f}",
        textposition="outside",
        hovertemplate="%{x}<br>План: %{y:,.0f} ₽<extra></extra>",
    ),
    secondary_y=False,
)
fig.add_trace(
    go.Bar(
        x=табл["неделя_метка"],
        y=табл["факт_сумма"],
        name="Факт, ₽",
        marker_color="#4169E1",  # синий
        text=табл["факт_сумма_окр"],
        texttemplate="%{text:,.0f}",
        textposition="outside",
        hovertemplate="%{x}<br>Факт: %{y:,.0f} ₽<extra></extra>",
    ),
    secondary_y=False,
)

# линия: % выполнения за неделю
positions = ["top right" if i % 2 == 0 else "top left" for i in range(len(табл))]
fig.add_trace(
    go.Scatter(
        x=табл["неделя_метка"],
        y=табл["выполнение_%"],
        mode="lines+markers+text",
        name="% выполнения (неделя)",
        line=dict(color="#DC143C", width=2),
        marker=dict(color="#DC143C"),
        text=табл["выполнение_%_окр"].astype(str) + "%",
        textposition=positions,
        textfont=dict(size=12),
        cliponaxis=False,
        hovertemplate="%{x}<br>Выполнение: %{y:.1f}%<extra></extra>",
    ),
    secondary_y=True,
)

fig.update_layout(
    title=f"План-факт по неделям ({месяц_план}), план {план_месяца:,.0f} ₽",
    barmode="group",
    bargap=0.25,
    hovermode="x unified",
    legend=dict(orientation="h", yanchor="bottom", y=-0.25, xanchor="center", x=0.5),
    margin=dict(t=80, r=40, b=80, l=70),
)

# ось X — только недели месяца
fig.update_xaxes(type="category", categoryorder="array", categoryarray=cats, title_text="Неделя (понедельник)")
# оси Y
fig.update_yaxes(title_text="Сумма, ₽", tickformat=",", secondary_y=False)
fig.update_yaxes(title_text="% выполнения", tickformat=",.0f", secondary_y=True)

fig.show()

# (опционально) сохранение
имя = f"план_факт_недели_{месяц_план}.html"
fig.write_html(имя, include_plotlyjs="cdn")
print(f"Сохранено: {имя}")

# ------ (необязательно) печать сводной таблицы ------
отчёт = табл[["неделя_метка", "план_сумма_окр", "факт_сумма_окр", "выполнение_%_окр"]]
отчёт.columns = ["неделя", "план_руб", "факт_руб", "выполнение_%"]
print(отчёт.to_string(index=False, formatters={"план_руб": "{:,.0f}".format, "факт_руб": "{:,.0f}".format}))


Сохранено: план_факт_недели_2025-10.html
    неделя  план_руб  факт_руб  выполнение_%
2025-09-29 2,822,581 1,910,000         67.70
2025-10-06 3,951,613 3,610,400         91.40
2025-10-13 3,951,613    90,000          2.30
2025-10-20 3,951,613         0          0.00
2025-10-27 2,822,581         0          0.00


In [6]:
df.to_excel('свод.xlsx', index=False)